# 3. BB84, end to end

The course's quantum key distribution protocol, with the quantum bit error
rate measured in all three scenarios.


In [ ]:
import math
import sys
from pathlib import Path

# Make the shared qc101 library importable, whether the notebook is opened
# from the repository root or from inside the notebooks/ directory.
_here = Path.cwd().resolve()
for _candidate in (_here / "labs" / "common",
                   _here.parent / "labs" / "common",
                   _here / ".." / "labs" / "common"):
    if (_candidate / "qc101").is_dir():
        sys.path.insert(0, str(_candidate.resolve()))
        break
else:
    raise SystemExit("Could not find the qc101 library -- run this notebook "
                     "from the repository root or from notebooks/.")

from qc101 import boolean, complexnum, linalg, photon, probability, quantum, viz
from qc101.labkit import Lab

print("qc101 library ready")


## The protocol

Each round: Alice picks a bit and a basis, Bob picks a basis, they compare
bases publicly and keep the matching rounds.


In [ ]:
from qc101 import bb84

for label, kwargs in (("no eavesdropper", {"eve": False}),
                       ("eve, same basis ", {"eve": True, "eve_resend_mode": "same_basis"}),
                       ("eve, random basis", {"eve": True, "eve_resend_mode": "random_basis"})):
    result = bb84.run_bb84(rounds=2000, seed=2024, **kwargs)
    print(f"{label}:  sifted {result.sifted_length:>3}  "
          f"QBER {result.qber * 100:5.2f}%  "
          f"detected: {result.detected}  keys match: {result.key_matches}")

## Why 25%?

Half the sifted rounds used the same basis (no error) and half used
different bases (half of *those* are wrong):

$$Q = \tfrac12 \cdot 0 + \tfrac12 \cdot \tfrac12 = \tfrac14$$


In [ ]:
print("no eavesdropper       :   0.00%")
print("eve, same basis        : ~ 25.00%   (the textbook intercept-resend)")
print("eve, random basis      : ~ 37.50%   (a noisier but stronger attack)")
print("one-way abort threshold: ~ 11.00%")
print()
print("the threshold used by the library:", bb84.QBER_THRESHOLD)

## The one-time pad

$C = M \oplus K$ and $M = C \oplus K$.  With a one-time, uniformly random,
equal-length key this is unconditionally secure.


In [ ]:
import random
rng = random.Random(99)

message = "ATTACK AT DAWN"
cipher, pad = boolean.one_time_pad_encrypt(message, rng)
plain = boolean.one_time_pad_decrypt(cipher, pad)
print("message  :", message)
print("pad bits :", "".join(map(str, pad[:32])), "...")
print("cipher   :", "".join(map(str, cipher[:32])), "...")
print("decrypted:", plain)
print("round trip works:", plain == message)